In [1]:
# 0_FNS

from pathlib import Path
from difflib import get_close_matches

import pandas as pd


# -----######-----###### ONE-FOLDER LEGAL EXHIBIT -----######-----######

def _legalbal_2007_i2_GET_one_folder(
    df: pd.DataFrame,
    folder_name: str,
    reference_date: str = "2026-07-01",
    output_folder: str | Path | None = None,
    folder_id_map: dict | None = None,
):
    """
    Create one legal balance exhibit for one account folder.

    Parameters
    ----------
    df : pd.DataFrame
        Master dataframe containing account_uid, date, and balance.

    folder_name : str
        Exact folder name from FOLDER_ID_MAP.

        Example:
            "Chase_Freedom_Visa_x3262"

    reference_date : str
        Reporting date. It is forced to the first day of the month.

    output_folder : str | Path | None
        Folder where the legal workbook will be saved.

    folder_id_map : dict | None
        Folder registry dictionary. Uses global FOLDER_ID_MAP
        when omitted.

    Returns
    -------
    tuple
        df_legal_summary
        df_legal_details
        output_path
    """

    if folder_id_map is None:
        folder_id_map = globals().get(
            "FOLDER_ID_MAP",
            {},
        )

    if not folder_id_map:
        raise ValueError(
            "FOLDER_ID_MAP is missing or empty."
        )

    clean_folder_name = str(
        folder_name
    ).strip()

    # Case-insensitive folder lookup.
    normalized_lookup = {
        str(folder).strip().casefold(): folder
        for folder in folder_id_map
    }

    matched_folder = normalized_lookup.get(
        clean_folder_name.casefold()
    )

    if matched_folder is None:
        close_matches = get_close_matches(
            clean_folder_name,
            list(folder_id_map.keys()),
            n=5,
            cutoff=0.45,
        )

        suggestion_text = (
            "\nPossible matches:\n- "
            + "\n- ".join(close_matches)
            if close_matches
            else ""
        )

        raise ValueError(
            f"Folder not found in FOLDER_ID_MAP: "
            f"{clean_folder_name}"
            f"{suggestion_text}"
        )

    registry_record = folder_id_map[
        matched_folder
    ]

    if not isinstance(registry_record, dict):
        raise ValueError(
            f"Registry entry for '{matched_folder}' "
            "must be a dictionary."
        )

    registry_id = registry_record.get(
        "registry_id"
    )

    account_uid = registry_record.get(
        "account_uid"
    )

    if not account_uid:
        raise ValueError(
            f"Folder '{matched_folder}' has no account_uid."
        )

    reference_timestamp = pd.to_datetime(
        reference_date,
        errors="coerce",
    )

    if pd.isna(reference_timestamp):
        raise ValueError(
            "reference_date must be a valid date."
        )

    reference_timestamp = (
        reference_timestamp
        .normalize()
        .replace(day=1)
    )

    reference_string = (
        reference_timestamp.strftime("%Y-%m-%d")
    )

    registry_prefix = (
        str(registry_id).strip()
        if registry_id
        else "NO_REGISTRY"
    )

    output_name = (
        f"{registry_prefix}_"
        f"{account_uid}_"
        f"LEGAL_BALANCE_EXHIBIT_"
        f"{reference_string}.xlsx"
    )

    account_rows = df.loc[
        df["account_uid"]
        .astype("string")
        .str.strip()
        .eq(account_uid)
    ]

    print("\n" + "=" * 75)
    print("ONE-ACCOUNT LEGAL EXHIBIT")
    print("=" * 75)
    print(f"Folder:       {matched_folder}")
    print(f"Registry ID:  {registry_id}")
    print(f"Account UID:  {account_uid}")
    print(f"Rows found:   {len(account_rows):,}")
    print(f"As-of date:   {reference_string}")
    print("=" * 75)

    df_summary, df_details, output_path = (
        _legalbal_2007_i1_GET_exhibit(
            df=df,
            uid_list=[account_uid],
            reference_date=reference_string,
            output_folder=output_folder,
            output_name=output_name,
            exhibit_title=(
                "ACCOUNT BALANCE SNAPSHOT EXHIBIT"
            ),
            case_caption=(
                f"Financial account balance history for "
                f"{account_uid}, as of "
                f"{reference_timestamp:%B %d, %Y}, "
                "with comparative balances three months, "
                "six months, and one year earlier."
            ),
            folder_id_map=folder_id_map,
        )
    )

    return (
        df_summary,
        df_details,
        output_path,
    )

In [3]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!
from fns import  _xlsx_1607_i3_GET_all_accounts

df_all = _xlsx_1607_i3_GET_all_accounts(
    data_folder="data",
    file_template="finances_{year}.xlsx",
    start_year=2021,
    end_year=2026,
    output_filename="finances_2021_2026.xlsx",
    output_sheet="ALL_TRANSACTIONS",
)

Processing yearly workbooks:   0%|          | 0/6 [00:00<?, ?workbook/s]


finances_2021.xlsx
Account sheets found: ['PayPal', 'OUCU Checking', 'OUCU Visa', 'PNC Joint']


Reading finances_2021.xlsx:   0%|          | 0/4 [00:00<?, ?sheet/s]

ADDED: PayPal | 40 rows
ADDED: OUCU Checking | 576 rows
ADDED: OUCU Visa | 36 rows
ADDED: PNC Joint | 6 rows

finances_2022.xlsx
Account sheets found: ['PayPal', 'OUCU Checking', 'OUCU Visa', 'PNC Joint', 'Robinhood']


Reading finances_2022.xlsx:   0%|          | 0/5 [00:00<?, ?sheet/s]

ADDED: PayPal | 83 rows
ADDED: OUCU Checking | 211 rows
ADDED: OUCU Visa | 25 rows
ADDED: PNC Joint | 565 rows
ADDED: Robinhood | 53 rows

finances_2023.xlsx
Account sheets found: ['PayPal', 'OUCU Checking', 'OUCU Visa', 'PNC Joint', 'Robinhood']


... (4 outputs left out)

Reading finances_2025.xlsx:   0%|          | 0/7 [00:00<?, ?sheet/s]

ADDED: PayPal | 653 rows
ADDED: OUCU Checking | 1,158 rows
ADDED: OUCU Visa | 94 rows
ADDED: Venmo | 87 rows
ADDED: PNC Joint | 23 rows
ADDED: Robinhood | 48 rows
ADDED: Cash App | 16 rows

finances_2026.xlsx
Account sheets found: ['PayPal', 'OUCU Checking', 'OUCU Visa', 'Venmo', 'PNC Joint', 'Robinhood', 'Chase Checking', 'Chase Savings', 'Chase Freedom']


Reading finances_2026.xlsx:   0%|          | 0/9 [00:00<?, ?sheet/s]

ADDED: PayPal | 116 rows
ADDED: OUCU Checking | 543 rows
ADDED: OUCU Visa | 24 rows
ADDED: Venmo | 15 rows
ADDED: PNC Joint | 10 rows
ADDED: Robinhood | 45 rows
ADDED: Chase Checking | 38 rows
ADDED: Chase Savings | 3 rows
ADDED: Chase Freedom | 28 rows

================ COMPLETE ================
Account sheets combined: 35
Total rows: 6,488
Total columns: 24
Output sheet count: 1
Saved to: /Users/yerik/_apple_lib/_a_progs/_a0ds_env/__finances/_2_accounts /checkings/data/finances_2021_2026.xlsx


In [4]:
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!

from IPython.display import display


df_one_summary, df_one_details, one_exhibit_path = (
    _legalbal_2007_i2_GET_one_folder(
        df=df_all,
        folder_name="Chase_Freedom_Visa_x3262",
        reference_date="2026-07-01",
        output_folder=(
            "/Users/yerik/Documents/"
            "___Divorce_2nd_Paper/"
            "__inbox/ACCOUNT_BALANCE_EXPORTS"
        ),
        folder_id_map=FOLDER_ID_MAP,
    )
)


balance_columns = [
    column
    for column in df_one_summary.columns
    if column.startswith("Balance as of")
]


display(
    df_one_summary.style
    .hide(axis="index")
    .format(
        {
            column: "${:,.2f}"
            for column in balance_columns
        },
        na_rep="Not available",
    )
)


display(
    df_one_details[
        [
            "Period",
            "Requested As-of Date",
            "Balance Date Used",
            "Balance",
            "Record Method",
            "Record Note",
        ]
    ].style
    .hide(axis="index")
    .format(
        {
            "Requested As-of Date": "{:%m/%d/%Y}",
            "Balance Date Used": lambda value: (
                value.strftime("%m/%d/%Y")
                if pd.notna(value)
                else "Not available"
            ),
            "Balance": "${:,.2f}",
        },
        na_rep="Not available",
    )
)


print(f"\nSaved legal exhibit:\n{one_exhibit_path}")

NameError: name 'FOLDER_ID_MAP' is not defined